# LeetCode 3448 — Count Substrings Divisible By Last Digit bằng Python

- Link: https://leetcode.com/problems/count-substrings-divisible-by-last-digit/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động theo số dư** (`cnt[d][r]` cho mọi `d = 1..9`), **dấu hiệu chia hết**, **đếm theo tiền tố** (mod 3, 9, 7 với nghịch đảo của 10)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countSubstrings(self, s: str) -> int:
```

## 1. Mục tiêu học

1. Đếm xâu con theo **điểm kết thúc**: xâu con kết thúc tại `i` có chữ số cuối `x = s[i]`; chỉ cần biết giá trị của nó **mod `x`**.
2. DP: `cnt[d][r]` = số xâu con kết thúc tại vị trí hiện tại có giá trị `≡ r (mod d)`, cho **mọi** `d = 1..9` cùng lúc; chuyển trạng thái `r → (10r + x) mod d`.
3. Cách tuyến tính thuần túy bằng **dấu hiệu chia hết**: 1, 2, 5 (luôn đúng), 4 và 8 (2–3 chữ số cuối), 3, 6, 9 (tổng chữ số — đếm tiền tố), 7 (tiền tố mod 7 nhân nghịch đảo lũy thừa 10).
4. Xử lý chữ số cuối bằng 0 (không tính) và số 0 đứng đầu.
5. Kiểm chứng với vét cạn `O(n²)` (theo mod 2520 = BCNN(1..9)).

## 2. Đề bài & ví dụ

Cho xâu chữ số `s`. Đếm số xâu con (theo vị trí) **chia hết cho chữ số cuối khác 0** của nó. Xâu con có thể có số 0 đứng đầu; xâu con kết thúc bằng `0` không được tính.

**Ví dụ 1:** `"12936"` → `11`. **Ví dụ 2:** `"5701283"` → `18`. **Ví dụ 3:** `"1010101010"` → `25`.

**Ràng buộc:** `1 ≤ |s| ≤ 10^5`.

In [1]:
examples = [("12936", 11), ("5701283", 18), ("1010101010", 25)]
for s, exp in examples:
    print(s, '-> kỳ vọng', exp)

12936 -> kỳ vọng 11
5701283 -> kỳ vọng 18
1010101010 -> kỳ vọng 25


## 3. Kiến thức nền

### 3.1. Số dư khi nối thêm một chữ số

Nếu `v ≡ r (mod d)` thì `10v + x ≡ 10r + x (mod d)`. Đọc xâu từ trái sang, số dư cập nhật được mà không cần số lớn.

In [2]:
def mod_of(sub, d):
    r = 0
    for ch in sub:
        r = (r * 10 + ord(ch) - 48) % d
    return r

print(mod_of("2936", 6), int("2936") % 6, mod_of("0128", 8), int("0128") % 8)

2 2 0 0


### 3.2. BCNN(1..9) = 2520

Giữ giá trị mod 2520 là đủ để biết chia hết cho mọi chữ số 1..9 (vì mọi `d ≤ 9` chia hết 2520).

### 3.3. Dấu hiệu chia hết

- `1, 2, 5`: chữ số cuối là chính nó ⇒ **luôn** chia hết.
- `4`: hai chữ số cuối chia hết cho 4; `8`: ba chữ số cuối chia hết cho 8.
- `3, 9`: tổng chữ số chia hết cho 3 (9). `6`: chẵn (đúng sẵn vì cuối là 6) **và** chia hết cho 3.
- `7`: không có dấu hiệu đơn giản ⇒ dùng tiền tố mod 7 (mục 7.2).

In [3]:
from math import lcm
from functools import reduce
print('BCNN(1..9) =', reduce(lcm, range(1, 10)))
print('các số 2 chữ số kết thúc bằng 4 chia hết cho 4:', [v for v in range(4, 100, 10) if v % 4 == 0])
print('các số 3 chữ số kết thúc bằng 8 chia hết cho 8:', len([v for v in range(8, 1000, 10) if v % 8 == 0]), 'trên 100')

BCNN(1..9) = 2520
các số 2 chữ số kết thúc bằng 4 chia hết cho 4: [4, 24, 44, 64, 84]
các số 3 chữ số kết thúc bằng 8 chia hết cho 8: 25 trên 100


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `"12936"`: có 15 xâu con, tất cả chữ số cuối khác 0.

- Kết thúc bằng `1`, `2`: luôn đúng ⇒ `1 + 2 = 3`.
- Kết thúc bằng `9` (vị trí 2): `"9"` ✓, `"29"` (tổng 11) ✗, `"129"` (tổng 12) ✗ ⇒ 1.
- Kết thúc bằng `3` (vị trí 3): `"3"`, `"93"` (12) ✓, `"293"` (14) ✗, `"1293"` (15) ✓ ⇒ 3.
- Kết thúc bằng `6` (vị trí 4): `"6"`, `"36"`, `"936"`, `"2936"` (tổng 20 ✗), `"12936"` (21) ✓ ⇒ 4.

Tổng `3 + 1 + 3 + 4 = 11` ✓.

In [4]:
s = "12936"
for i in range(len(s)):
    x = int(s[i])
    ok = [s[j:i + 1] for j in range(i + 1) if int(s[j:i + 1]) % x == 0]
    print(f"kết thúc tại {i} (chữ số {x}): {ok}")

kết thúc tại 0 (chữ số 1): ['1']
kết thúc tại 1 (chữ số 2): ['12', '2']
kết thúc tại 2 (chữ số 9): ['9']
kết thúc tại 3 (chữ số 3): ['1293', '93', '3']
kết thúc tại 4 (chữ số 6): ['12936', '936', '36', '6']


## 5. Cách 1 — Brute Force: mọi điểm bắt đầu, cập nhật số dư mod 2520

In [5]:
def count_brute(s):
    n, c = len(s), 0
    for i in range(n):
        v = 0
        for j in range(i, n):
            x = ord(s[j]) - 48
            v = (v * 10 + x) % 2520
            if x and v % x == 0:
                c += 1
    return c

for s, exp in examples:
    print(count_brute(s), '| kỳ vọng', exp)

11 | kỳ vọng 11
18 | kỳ vọng 18
25 | kỳ vọng 25


**Độ phức tạp:** `O(n²)` — `5·10^9` bước với `n = 10^5`: quá chậm.

## 6. Ý tưởng thuật toán / bước đột phá

Duy trì cho **mỗi** `d = 1..9` mảng `cnt[d][0..d−1]`: số xâu con kết thúc tại vị trí hiện tại có giá trị `≡ r (mod d)`. Khi đọc chữ số `x`:

- mọi xâu con cũ được nối thêm `x`: `new[d][(10r + x) mod d] += cnt[d][r]`;
- xâu con mới chỉ gồm `x`: `new[d][x mod d] += 1`.

Nếu `x > 0`, cộng `new[x][0]` vào đáp án. Mỗi vị trí tốn `Σ d = 45` phép ⇒ `O(45 n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP số dư cho mọi `d` (lời giải chính, theo gợi ý)

In [6]:
def count_dp(s):
    cnt = [[0] * d for d in range(10)]        # cnt[0] không dùng
    ans = 0
    for ch in s:
        x = ord(ch) - 48
        new = [None]
        for d in range(1, 10):
            old, nw = cnt[d], [0] * d
            for r in range(d):
                if old[r]:
                    nw[(r * 10 + x) % d] += old[r]
            nw[x % d] += 1
            new.append(nw)
        cnt = new
        if x:
            ans += cnt[x][0]
    return ans

for s, exp in examples:
    print(count_dp(s), '| kỳ vọng', exp)

11 | kỳ vọng 11
18 | kỳ vọng 18
25 | kỳ vọng 25


### 7.2. Cách 3 — Dấu hiệu chia hết + đếm tiền tố, `O(n)`

- `x ∈ {1, 2, 5}`: cộng `i + 1` (mọi xâu con kết thúc tại `i`).
- `x ∈ {3, 6}`: xâu con `s[j..i]` có tổng chữ số `≡ 0 (mod 3)` ⇔ hai **tổng tiền tố** `S[i+1] ≡ S[j] (mod 3)` ⇒ đếm số tiền tố cùng số dư. Với `6`, tính chẵn đã đúng sẵn.
- `x = 9`: tương tự với mod 9.
- `x = 4`: `"4"` luôn đúng; độ dài ≥ 2 đúng ⇔ `10·s[i−1] + 4 ≡ 0 (mod 4)` ⇒ cộng `i` (mọi điểm bắt đầu `j ≤ i − 1`).
- `x = 8`: `"8"` đúng; độ dài 2 đúng ⇔ `10·s[i−1] + 8 ≡ 0 (mod 8)`; độ dài ≥ 3 đúng ⇔ ba chữ số cuối chia hết cho 8 ⇒ cộng `i − 1`.
- `x = 7`: với tiền tố `P[k]` (giá trị mod 7 của `s[0..k)`), giá trị `s[j..i] ≡ P[i+1] − P[j] · 10^{i+1−j}`. Vì `10` khả nghịch mod 7 (`10^{-1} ≡ 5`), điều kiện `≡ 0` ⇔ `P[i+1] · 5^{i+1} ≡ P[j] · 5^{j} (mod 7)` ⇒ đếm khóa `P[k] · 5^k mod 7`.

In [7]:
def count_rules(s):
    d = [ord(c) - 48 for c in s]
    ans = 0
    c3, c9 = [1, 0, 0], [1] + [0] * 8           # tiền tố rỗng
    ps3 = ps9 = 0
    c7 = [1] + [0] * 6
    P7, inv_pow = 0, 1
    for i, x in enumerate(d):
        ps3 = (ps3 + x) % 3
        ps9 = (ps9 + x) % 9
        P7 = (P7 * 10 + x) % 7
        inv_pow = inv_pow * 5 % 7                 # 5 = 10^{-1} mod 7
        key = P7 * inv_pow % 7
        if x in (1, 2, 5):
            ans += i + 1
        elif x in (3, 6):
            ans += c3[ps3]
        elif x == 9:
            ans += c9[ps9]
        elif x == 4:
            ans += 1 + (i if i >= 1 and (d[i - 1] * 10 + 4) % 4 == 0 else 0)
        elif x == 8:
            ans += 1
            if i >= 1 and (d[i - 1] * 10 + 8) % 8 == 0:
                ans += 1
            if i >= 2 and (d[i - 2] * 100 + d[i - 1] * 10 + 8) % 8 == 0:
                ans += i - 1
        elif x == 7:
            ans += c7[key]
        c3[ps3] += 1
        c9[ps9] += 1
        c7[key] += 1
    return ans

for s, exp in examples:
    print(count_rules(s), '| kỳ vọng', exp)

11 | kỳ vọng 11
18 | kỳ vọng 18
25 | kỳ vọng 25


Lưu ý thứ tự: khi xét vị trí `i`, các bộ đếm `c3, c9, c7` chứa các tiền tố `0..i` (chưa có tiền tố `i + 1`), còn `ps3, ps9, key` là của tiền tố `i + 1`. Cặp `(j, i + 1)` với `j ≤ i` chính là các xâu con `s[j..i]`.

### 7.3. Cách 4 — DP số dư dùng bảng chuyển tính sẵn (tăng tốc cách 2)

Với mỗi `(d, x)`, tính sẵn hoán vị `r → (10r + x) mod d`. Khi đó mảng mới chỉ là "dồn" các ô theo bảng, không cần phép `%` trong vòng lặp nóng.

In [8]:
TRANS = [[None] * 10 for _ in range(10)]
for d in range(1, 10):
    for x in range(10):
        TRANS[d][x] = [(r * 10 + x) % d for r in range(d)]

def count_dp_table(s):
    cnt = [None] + [[0] * d for d in range(1, 10)]
    ans = 0
    for ch in s:
        x = ord(ch) - 48
        for d in range(1, 10):
            old, tr = cnt[d], TRANS[d][x]
            nw = [0] * d
            for r in range(d):
                nw[tr[r]] += old[r]
            nw[x % d] += 1
            cnt[d] = nw
        if x:
            ans += cnt[x][0]
    return ans

for s, exp in examples:
    print(count_dp_table(s), '| kỳ vọng', exp)

11 | kỳ vọng 11
18 | kỳ vọng 18
25 | kỳ vọng 25


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: mỗi vị trí in chữ số, quy tắc dùng và số xâu con được cộng.

In [9]:
def trace_rules(s):
    d = [ord(c) - 48 for c in s]
    ans, c3, ps3 = 0, [1, 0, 0], 0
    c9, ps9 = [1] + [0] * 8, 0
    for i, x in enumerate(d):
        ps3 = (ps3 + x) % 3; ps9 = (ps9 + x) % 9
        if x in (1, 2, 5):
            add, why = i + 1, 'luôn chia hết'
        elif x in (3, 6):
            add, why = c3[ps3], f'tiền tố có tổng ≡ {ps3} (mod 3): {c3[ps3]} cái'
        elif x == 9:
            add, why = c9[ps9], f'tiền tố có tổng ≡ {ps9} (mod 9): {c9[ps9]} cái'
        else:
            add, why = None, 'xem cách 3'
        c3[ps3] += 1; c9[ps9] += 1
        if add is not None:
            ans += add
        print(f"i={i} chữ số {x}: {why} -> +{add} (tổng {ans})")
    print('kết quả:', ans)

trace_rules("12936")

i=0 chữ số 1: luôn chia hết -> +1 (tổng 1)
i=1 chữ số 2: luôn chia hết -> +2 (tổng 3)
i=2 chữ số 9: tiền tố có tổng ≡ 3 (mod 9): 1 cái -> +1 (tổng 4)
i=3 chữ số 3: tiền tố có tổng ≡ 0 (mod 3): 3 cái -> +3 (tổng 7)
i=4 chữ số 6: tiền tố có tổng ≡ 0 (mod 3): 4 cái -> +4 (tổng 11)
kết quả: 11


## 9. Code nộp LeetCode

Dùng cách 3 (tuyến tính, nhanh nhất trong Python).

In [10]:
class Solution:
    def countSubstrings(self, s: str) -> int:
        ans = 0
        c3 = [1, 0, 0]
        c9 = [1, 0, 0, 0, 0, 0, 0, 0, 0]
        c7 = [1, 0, 0, 0, 0, 0, 0]
        ps3 = ps9 = p7 = 0
        inv_pow = 1
        prev1 = prev2 = 0                      # s[i-1], s[i-2]
        for i, ch in enumerate(s):
            x = ord(ch) - 48
            ps3 = (ps3 + x) % 3
            ps9 = (ps9 + x) % 9
            p7 = (p7 * 10 + x) % 7
            inv_pow = inv_pow * 5 % 7
            key = p7 * inv_pow % 7
            if x == 1 or x == 2 or x == 5:
                ans += i + 1
            elif x == 3 or x == 6:
                ans += c3[ps3]
            elif x == 9:
                ans += c9[ps9]
            elif x == 4:
                ans += 1
                if i >= 1 and (prev1 * 10 + 4) % 4 == 0:
                    ans += i
            elif x == 8:
                ans += 1
                if i >= 1 and (prev1 * 10 + 8) % 8 == 0:
                    ans += 1
                if i >= 2 and (prev2 * 100 + prev1 * 10 + 8) % 8 == 0:
                    ans += i - 1
            elif x == 7:
                ans += c7[key]
            c3[ps3] += 1
            c9[ps9] += 1
            c7[key] += 1
            prev2, prev1 = prev1, x
        return ans

for s, exp in examples:
    got = Solution().countSubstrings(s)
    print(got, 'OK' if got == exp else 'SAI')

11 OK
18 OK
25 OK


## 10. Vì sao thuật toán đúng

**Cách 2 (DP):** bất biến — sau khi đọc `s[0..i]`, `cnt[d][r]` bằng số `j ≤ i` với `int(s[j..i]) ≡ r (mod d)`. Đúng với `i = 0`; bước chuyển: mỗi `s[j..i]` (`j ≤ i`) là `s[j..i−1]` nối `x` (số dư biến đổi `r → 10r + x`) hoặc chính `x` (`j = i`). Đáp án tại `i` là `cnt[x][0]`.

**Cách 3 (quy tắc):**

1. `x ∈ {1, 2, 5}`: `v ≡ x (mod 10)` ⇒ `v` chia hết cho `x` (vì `x | 10` và `x | x`).
2. `x ∈ {3, 9}`: `v ≡ tổng chữ số (mod 3, mod 9)`. `x = 6`: `v` chẵn (chữ số cuối chẵn) và cần chia hết cho 3.
3. `x = 4`: `100 ≡ 0 (mod 4)` ⇒ chỉ hai chữ số cuối quyết định. `x = 8`: `1000 ≡ 0 (mod 8)` ⇒ ba chữ số cuối quyết định; xâu con độ dài 2 thì chính nó.
4. `x = 7`: `gcd(10, 7) = 1` ⇒ nhân hai vế với `10^{-(i+1)}` là phép biến đổi tương đương.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mod 2520 | `O(n²)` | `O(1)` |
| 2. DP `cnt[d][r]` | `O(45 n)` | `O(45)` |
| 3. Dấu hiệu chia hết + tiền tố | `O(n)` | `O(1)` |
| 4. DP + bảng chuyển | `O(45 n)` (hằng số nhỏ hơn) | `O(450)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'dp': count_dp, 'rules': count_rules, 'dp_table': count_dp_table,
         'submit': lambda s: Solution().countSubstrings(s)}
edge = [("0", 0), ("7", 1), ("00007", 5), ("14", 2), ("104", 4), ("888", 6), ("77", 3), ("98", 2)]
for s, exp in examples + edge:
    assert count_brute(s) == exp, (s, count_brute(s))
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3448)
cnt = 0
for _ in range(3000):
    alpha = rng.choice(['0123456789', '0478', '0379', '0067'])
    s = ''.join(rng.choice(alpha) for _ in range(rng.randint(1, 40)))
    exp = count_brute(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'xâu ngẫu nhiên (có bộ chữ số tập trung vào 4, 7, 8, 9).')

Khớp vét cạn trên 3000 xâu ngẫu nhiên (có bộ chữ số tập trung vào 4, 7, 8, 9).


### 12.3. Ca lớn `n = 10^5`

In [13]:
import time
big = {
    'ngẫu nhiên': ''.join(rng.choice('0123456789') for _ in range(10 ** 5)),
    'toàn 7': '7' * 10 ** 5,
    'toàn 8': '8' * 10 ** 5,
    'xen 0 và 9': '09' * (5 * 10 ** 4),
}
for name, s in big.items():
    t = time.time(); a = Solution().countSubstrings(s); t1 = time.time() - t
    t = time.time(); b = count_dp(s); t2 = time.time() - t
    assert a == b
    print(f"{name:12s}: {a:>12d} | quy tắc {t1:.2f}s | DP {t2:.2f}s")

ngẫu nhiên  :   2325043827 | quy tắc 0.04s | DP 0.33s


toàn 7      :   5000050000 | quy tắc 0.03s | DP 0.28s


toàn 8      :   5000050000 | quy tắc 0.04s | DP 0.29s


xen 0 và 9  :   2500050000 | quy tắc 0.03s | DP 0.24s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính cả xâu con kết thúc bằng `0` (coi `v % 0` là "chia hết" khi `v = 0`)

In [14]:
def wrong_zero(s):
    n, c = len(s), 0
    for i in range(n):
        v = 0
        for j in range(i, n):
            x = ord(s[j]) - 48
            v = (v * 10 + x) % 2520
            if x == 0:
                c += v == 0                      # SAI: chữ số cuối phải khác 0
            elif v % x == 0:
                c += 1
    return c

print('sai :', wrong_zero("1010101010"), '| đúng:', Solution().countSubstrings("1010101010"))

sai : 30 | đúng: 25


**Vì sao sai:** đề chỉ xét chữ số cuối **khác 0**; xâu con kết thúc bằng 0 bị loại hoàn toàn.

### Lỗi 2 — Dấu hiệu chia hết cho 4/8 chỉ nhìn chữ số cuối

In [15]:
def wrong_last_digit_only(s):
    ans = 0
    for i, ch in enumerate(s):
        x = ord(ch) - 48
        if x in (1, 2, 4, 5, 8):                 # SAI: 4 và 8 không "luôn" chia hết
            ans += i + 1
        elif x:
            ans += sum(1 for j in range(i + 1) if int(s[j:i + 1]) % x == 0)
    return ans

print('sai :', wrong_last_digit_only("14"), '| đúng:', Solution().countSubstrings("14"))

sai : 3 | đúng: 2


**Vì sao sai:** `14` kết thúc bằng 4 nhưng không chia hết cho 4. Chỉ `1, 2, 5` (ước của 10) mới "luôn đúng".

### Lỗi 3 — Dùng tổng chữ số cho 7

In [16]:
def wrong_seven(s):
    ans = 0
    for i, ch in enumerate(s):
        x = ord(ch) - 48
        if x == 7:
            ans += sum(1 for j in range(i + 1) if sum(map(int, s[j:i + 1])) % 7 == 0)   # SAI
        elif x:
            ans += sum(1 for j in range(i + 1) if int(s[j:i + 1]) % x == 0)
    return ans

print('sai :', wrong_seven("167"), '| đúng:', Solution().countSubstrings("167"), '| 167 % 7 =', 167 % 7, ', tổng chữ số =', 1 + 6 + 7)

sai : 4 | đúng: 3 | 167 % 7 = 6 , tổng chữ số = 14


**Vì sao sai:** dấu hiệu "tổng chữ số" chỉ đúng cho 3 và 9 vì `10 ≡ 1 (mod 3, mod 9)`; còn `10 ≡ 3 (mod 7)`. Ví dụ `167`: tổng chữ số 14 chia hết cho 7 nhưng `167 = 7·23 + 6`. Phải dùng tiền tố mod 7.

### Lỗi 4 — DP quên xâu con mới chỉ gồm chữ số hiện tại

In [17]:
def wrong_no_single(s):
    cnt = [[0] * d for d in range(10)]
    ans = 0
    for ch in s:
        x = ord(ch) - 48
        new = [None]
        for d in range(1, 10):
            nw = [0] * d
            for r in range(d):
                nw[(r * 10 + x) % d] += cnt[d][r]
            # SAI: thiếu nw[x % d] += 1
            new.append(nw)
        cnt = new
        if x:
            ans += cnt[x][0]
    return ans

print('sai :', wrong_no_single("12936"), '| đúng:', Solution().countSubstrings("12936"))

sai : 0 | đúng: 11


**Vì sao sai:** không bao giờ khởi tạo xâu con nào ⇒ mọi `cnt` luôn bằng 0.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm xâu con chia hết cho …" ⇒ đếm theo điểm kết thúc + số dư (DP hoặc tiền tố).
- Ước số nhỏ (≤ 9) ⇒ giữ số dư cho **mọi** ước cùng lúc (45 trạng thái).
- Mod `m` với `gcd(10, m) = 1` ⇒ tiền tố nhân nghịch đảo lũy thừa 10 ⇒ đếm cặp bằng nhau.
- Mod là ước của `10^k` ⇒ chỉ `k` chữ số cuối quyết định.

## 15. Biến thể của bài

1. **LC 2575 — Find the Divisibility Array of a String:** tiền tố chia hết cho `m`.
2. Chia hết cho **chữ số đầu** thay vì cuối ⇒ duyệt từ phải sang, DP tương tự với lũy thừa 10.
3. Chia hết cho một số `m` cố định (không phải chữ số) với `gcd(10, m) = 1` ⇒ một bảng đếm khóa `P · 10^{-k}`.
4. Chia hết cho **tổng** các chữ số ⇒ khó hơn nhiều (tổng thay đổi theo xâu con).

In [18]:
# Biến thể 3: đếm xâu con chia hết cho m cố định, gcd(10, m) = 1
def count_div_by_m(s, m):
    inv = pow(10, -1, m)
    cnt = {0: 1}
    P, ip, ans = 0, 1, 0
    for ch in s:
        P = (P * 10 + ord(ch) - 48) % m
        ip = ip * inv % m
        key = P * ip % m
        ans += cnt.get(key, 0)
        cnt[key] = cnt.get(key, 0) + 1
    return ans

def count_div_brute(s, m):
    return sum(1 for i in range(len(s)) for j in range(i + 1, len(s) + 1) if int(s[i:j]) % m == 0)

rng = random.Random(15)
for _ in range(300):
    s = ''.join(rng.choice('0123456789') for _ in range(rng.randint(1, 25)))
    m = rng.choice([3, 7, 11, 13, 17, 21, 99])
    assert count_div_by_m(s, m) == count_div_brute(s, m)
print('Biến thể 3 khớp vét cạn. "12936" chia hết cho 13:', count_div_by_m("12936", 13))

Biến thể 3 khớp vét cạn. "12936" chia hết cho 13: 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn mod 2520 | `O(n²)` | Oracle |
| 2 | DP số dư mọi `d` | `O(45 n)` | Tổng quát, đúng gợi ý |
| 3 | Dấu hiệu chia hết + tiền tố | `O(n)` | **Nộp bài** |
| 4 | DP + bảng chuyển | `O(45 n)` | Bản tăng tốc của 2 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "11111"`.

**Bài 2 (tính tay):** `s = "246"`.

**Bài 3 (điền TODO):** viết `divisibility_array(s, m)` (LC 2575).

**Bài 4 (tự viết lại):** biến thể 2 — chia hết cho **chữ số đầu** khác 0.

**Bài 5 (mini challenge):** đếm xâu con chia hết cho **cả** chữ số đầu **và** chữ số cuối (khác 0).

In [19]:
def divisibility_array_todo(s, m):
    # TODO
    raise NotImplementedError

def count_by_first_digit_todo(s):
    # TODO
    raise NotImplementedError

def count_by_first_and_last_todo(s):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chữ số cuối là 1. (2) Mọi số chia hết cho 1. (3) Số xâu con `= 15`.

**Bài 2.** (1) `"2"`, `"4"`, `"6"`, `"24"` (÷4 ✓), `"46"` (÷6? 46 lẻ chia 3 dư 1 ✗), `"246"` (÷6 ✓). (2) Đếm. (3) Kiểm tra bằng code.

**Bài 3.** (1) Tiền tố mod `m`. (2) `P = (10P + x) mod m`. (3) Trả `1` nếu `P = 0`.

**Bài 4.** (1) Cố định chữ số đầu `d` (9 lượt). (2) DP trái → phải như cách 2 nhưng chỉ khởi tạo xâu con mới khi gặp chữ số `d`. (3) Mỗi vị trí cộng `cnt[0]`; đối chiếu vét cạn mod 2520.

**Bài 5.** (1) Mod BCNN(đầu, cuối) ≤ 72. (2) Với mỗi cặp (đầu, cuối)… (3) Vét cạn mod 2520 là đủ cho kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().countSubstrings("11111")
a2 = Solution().countSubstrings("246")
assert a1 == 15 and a2 == count_brute("246") == 5
print('Bài 1:', a1, '| Bài 2:', a2)

def divisibility_array(s, m):
    out, P = [], 0
    for ch in s:
        P = (P * 10 + ord(ch) - 48) % m
        out.append(1 if P == 0 else 0)
    return out

assert divisibility_array("998244353", 3) == [1, 1, 0, 0, 0, 1, 1, 0, 0]       # ví dụ LC 2575
assert divisibility_array("1010", 10) == [0, 1, 0, 1]
print('Bài 3:', divisibility_array("998244353", 3))

Bài 1: 15 | Bài 2: 5
Bài 3: [1, 1, 0, 0, 0, 1, 1, 0, 0]


In [21]:
# Bài 4, 5
def count_by_first_digit(s):
    # Với mỗi chữ số đầu d: DP từ trái sang trên các xâu con BẮT ĐẦU bằng chữ số d.
    # cnt[r] = số xâu con như vậy kết thúc tại vị trí hiện tại có giá trị ≡ r (mod d).
    ans = 0
    for d in range(1, 10):
        cnt = [0] * d
        for ch in s:
            x = ord(ch) - 48
            nw = [0] * d
            for r in range(d):
                if cnt[r]:
                    nw[(r * 10 + x) % d] += cnt[r]
            if x == d:
                nw[x % d] += 1
            cnt = nw
            ans += cnt[0]
    return ans

def brute_first(s):
    c = 0
    for i in range(len(s)):
        x = ord(s[i]) - 48
        if x == 0:
            continue
        v = 0
        for j in range(i, len(s)):
            v = (v * 10 + ord(s[j]) - 48) % 2520
            if v % x == 0:
                c += 1
    return c

def count_by_first_and_last(s):
    c = 0
    for i in range(len(s)):
        a = ord(s[i]) - 48
        v = 0
        for j in range(i, len(s)):
            b = ord(s[j]) - 48
            v = (v * 10 + b) % 2520
            if a and b and v % a == 0 and v % b == 0:
                c += 1
    return c

rng = random.Random(19)
for _ in range(400):
    s = ''.join(rng.choice('0123456789') for _ in range(rng.randint(1, 25)))
    assert count_by_first_digit(s) == brute_first(s)
print('Bài 4 khớp vét cạn; "12936" ->', count_by_first_digit("12936"))
print('Bài 5 (vét cạn mod 2520): "12936" ->', count_by_first_and_last("12936"), '| "5701283" ->', count_by_first_and_last("5701283"))

Bài 4 khớp vét cạn; "12936" -> 12
Bài 5 (vét cạn mod 2520): "12936" -> 10 | "5701283" -> 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# DP tổng quát
cnt[d][r] với d = 1..9;  với chữ số x:
    new[d][(10r + x) % d] += cnt[d][r];  new[d][x % d] += 1
    if x: ans += new[x][0]
# Tuyến tính: 1,2,5 -> i+1;  3,6 -> c3[tổng%3];  9 -> c9[tổng%9]
#            4 -> 1 + i·[2 số cuối ÷ 4];  8 -> 1 + [2 số ÷ 8] + (i-1)·[3 số ÷ 8]
#            7 -> c7[P·5^(i+1) mod 7]
```

**Checklist:**

- [ ] Bỏ qua xâu con kết thúc bằng `0`?
- [ ] 4 và 8 dùng 2–3 chữ số cuối, không "luôn đúng"?
- [ ] Bộ đếm tiền tố khởi tạo với tiền tố rỗng (`c[0] = 1`)?
- [ ] 7 dùng nghịch đảo của 10 (`5`), không dùng tổng chữ số?
- [ ] Cập nhật bộ đếm **sau** khi cộng đáp án tại vị trí `i`?